# 52-Week High Proximity — Strategy Research Notebook

This notebook asks a specific research question:

> Does buying S&P 500 stocks closest to their trailing 252-day high and shorting those farthest below that high earn a positive return net of transaction costs when rebalanced monthly?

In plain English: the 52-week high is a salient price anchor. George and Hwang (2004) argue investors under-react when prices approach that anchor, so stocks near their high keep outperforming stocks that trade far below it. We test whether that ranking still produces a usable long/short spread after costs in our point-in-time S&P panel.

Example:

```text
It is the last trading day of March.

Stock A closed at $98; its 252-day high is $100  -> near_52w_high = 0.98 (near the top).
Stock B closed at $60; its 252-day high is $100  -> near_52w_high = 0.60 (near the bottom).

The strategy for April:
  LONG  the top 20% of the ranking  (stocks like A, closest to the high)
  SHORT the bottom 20% of the ranking (stocks like B, farthest from the high)

Positions are equal-weighted within each leg, held for one month, then re-ranked.
```

Registered as `near_52w_high` in `core/strategies/registry.py`. Factor column: `near_52w_high`. All math is imported from `core/`.

**Reference**: George & Hwang (2004) "The 52-Week High and Momentum Investing", Journal of Finance 59(5).

## 1. Setup And Data Loading

Paths resolve from the repository root whether Jupyter started in the root or inside `notebooks/`. Missing data files raise a clear error.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd if (_cwd / "core").is_dir() else _cwd.parent
assert (REPO_ROOT / "core").is_dir(), f"Could not locate repo root from {_cwd}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.backtest.portfolio import sp500_universe_filter
from core.data.factors.build_factors import proximity_to_52_week_high
from core.metrics.performance import (
    calculate_cumulative_returns,
    calculate_performance_metrics,
)
from core.strategies import get_strategy, run_factor_cross_section_backtest

FACTORS_PATH = REPO_ROOT / "data" / "factors" / "factors_price.parquet"
PRICES_PATH = REPO_ROOT / "data" / "factors" / "prices.parquet"

missing = [p for p in (FACTORS_PATH, PRICES_PATH) if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing data files: {[str(p) for p in missing]}. "
        "Run scripts/backfill_all.py (or restore data/factors/) before executing this notebook."
    )

factor_panel = pd.read_parquet(FACTORS_PATH)
adjusted_close_prices = pd.read_parquet(PRICES_PATH)
PANEL_TZ = adjusted_close_prices.index.tz

print("factor panel:", factor_panel.shape, "columns:", list(factor_panel.columns))
print("price panel: ", adjusted_close_prices.shape, "tz:", PANEL_TZ)
assert "near_52w_high" in factor_panel.columns

## 2. What Exactly Are We Observing?

The primary observed series is the **daily net return of a long/short equity portfolio**: arithmetic daily returns, equal-weighted within each leg, long minus short, minus transaction costs. Produced by `run_factor_cross_section_backtest`.

Input data sources:

- **Adjusted-close stock prices** (`data/factors/prices.parquet`): wide panel, daily, tz-aware `America/New_York`.
- **Factor panel** (`data/factors/factors_price.parquet`): MultiIndex `(date, symbol)` including `near_52w_high`.
- **Point-in-time S&P 500 membership** via `sp500_universe_filter()`.

The model does **not** see volume, news, analyst revisions, or unadjusted closes.

### Per-variable audit: `near_52w_high`

- **Source**: `core.data.factors.build_factors.proximity_to_52_week_high`.
- **Transformation**: `close / rolling_max(close, 252)` with `min_periods=252`.
- **Missing-data handling**: NaN until 252 closes exist; no fill.
- **Publication lag**: none beyond EOD prices (publication ≈ reference same evening).
- **Standardization timing**: none — raw ratio ranked cross-sectionally each rebalance.
- **Leakage risk**: low for prices; medium if adjusted history is restated after the as-of date (vendor restatement risk).

## 3. Signal Sanity Check

Recompute the factor on one raw price column and confirm it matches the panel.

In [ ]:
check_symbol = "AAPL"
recomputed = proximity_to_52_week_high(adjusted_close_prices[check_symbol], window=252)
panel_series = factor_panel.xs(check_symbol, level="symbol")["near_52w_high"]
aligned = pd.concat([recomputed.rename("recomputed"), panel_series.rename("panel")], axis=1).dropna()
max_abs_diff = float((aligned["recomputed"] - aligned["panel"]).abs().max())
print(f"{check_symbol}: rows={len(aligned)}, max |recomputed - panel| = {max_abs_diff:.3e}")
assert max_abs_diff < 1e-10
print("panel describe:")
print(factor_panel["near_52w_high"].describe())

## 4. Portfolio Construction Disclosure

There are no regimes or model-inferred labels. Construction is mechanical:

```text
Each month-end as_of_date:
  1. Restrict to names in the point-in-time S&P 500 on that date.
  2. Rank remaining names DESCENDING on near_52w_high (high = near high = long).
  3. Long top 20%, short bottom 20%, equal-weight within each leg.
  4. Execute with signal_lag_days=1 (T+1 close).
  5. Subtract transaction costs (default 10 bps one-way in the net run).
```

> When the strategy says "this name is a long", that means its `near_52w_high` was in the top quintile of eligible S&P names on the signal date — not that we forecasted a fundamental event.

In [ ]:
meta = get_strategy("near_52w_high")
print(meta.title)
print(meta.hypothesis)
print("expected Sharpe range:", meta.expected_sharpe_range)
print("known limitations:")
for line in meta.known_limitations:
    print(" -", line)

## 5. Backtest

Compare **net** 52-week-high L/S vs **gross** (0-cost) to isolate cost drag, and vs **12-1 momentum** as a correlated reference factor. Window starts in 2018 to reduce pre-FMP survivorship noise.

In [ ]:
BACKTEST_START = pd.Timestamp("2018-01-01", tz=PANEL_TZ)
BACKTEST_END = pd.Timestamp("2025-12-31", tz=PANEL_TZ)
TRANSACTION_COST = 0.001  # 10 bps
sp500_membership = sp500_universe_filter()

backtest_runs = {
    "near_52w_net_10bps": dict(factor_col="near_52w_high", transaction_cost=TRANSACTION_COST),
    "near_52w_gross": dict(factor_col="near_52w_high", transaction_cost=0.0),
    "mom_12_1_net_10bps": dict(factor_col="mom_12_1", transaction_cost=TRANSACTION_COST),
}

daily_net_returns = {}
for run_name, overrides in backtest_runs.items():
    daily_net_returns[run_name] = run_factor_cross_section_backtest(
        factor_panel,
        adjusted_close_prices,
        start=BACKTEST_START,
        end=BACKTEST_END,
        top_pct=0.20,
        bottom_pct=0.20,
        long_only=False,
        rebalance_freq="ME",
        min_stocks=20,
        signal_lag_days=1,
        universe_filter=sp500_membership,
        **overrides,
    )
    print(f"{run_name}: {len(daily_net_returns[run_name])} trading days")


In [ ]:
metrics_by_run = pd.DataFrame(
    {name: calculate_performance_metrics(rets) for name, rets in daily_net_returns.items()}
).T[
    [
        "annualized_return",
        "annualized_volatility",
        "sharpe_ratio",
        "sortino_ratio",
        "max_drawdown",
        "calmar_ratio",
    ]
]
metrics_by_run.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for run_name, rets in daily_net_returns.items():
    cum = calculate_cumulative_returns(rets)
    ax.plot(cum.index, cum.values, label=run_name, linewidth=1.4)
ax.set_title("Cumulative net returns: 52-week high vs momentum reference")
ax.set_ylabel("Growth of $1")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Decision

We compare the **net 52-week-high long/short** to **gross 52-week-high** (cost drag) and to **net 12-1 momentum** (correlated reference).

Sign convention for deltas: `near_net − reference`. Positive Sharpe delta vs momentum means the 52-week-high book beat momentum on risk-adjusted terms in this window.

Decision metrics (yes/no):

1. Is net Sharpe positive?
2. Does gross → net leave Sharpe still usable (not wiped by 10 bps)?
3. Does net Sharpe beat `mom_12_1` net in the same window?

In [ ]:
decision_metrics = ["sharpe_ratio", "annualized_return", "max_drawdown"]
near_net = metrics_by_run.loc["near_52w_net_10bps", decision_metrics]
near_gross = metrics_by_run.loc["near_52w_gross", decision_metrics]
mom_net = metrics_by_run.loc["mom_12_1_net_10bps", decision_metrics]

print("Net Sharpe positive?", bool(near_net["sharpe_ratio"] > 0))
print("Net Sharpe after costs still > 0.1?", bool(near_net["sharpe_ratio"] > 0.1))
print("Beats mom_12_1 net Sharpe?", bool(near_net["sharpe_ratio"] > mom_net["sharpe_ratio"]))
print()
print("near_net - near_gross (cost drag):")
print((near_net - near_gross).to_string())
print()
print("near_net - mom_net:")
print((near_net - mom_net).to_string())

## 7. Disclosed Risks And Limitations

- **Not independent of momentum**: proximity to highs overlaps intermediate past winners.
- **Momentum-crash co-movement**: sharp recoveries can punish both books together.
- **Adjusted-close highs**: split/dividend adjustments move the rolling max vs quote-screen highs traders see.
- **Survivorship / coverage**: FMP EOD gaps for many pre-~2015 delisted S&P members; this notebook uses 2018+ for that reason.
- **Costs**: monthly rebalance understates weekly implementations sometimes used in papers.
- **Single sample**: one US large-cap window is not out-of-sample proof.